# 04 · Ejecución QAOA en SelenePlus

Este notebook documenta exclusivamente la ejecución remota final de QAOA
Max-Cut en **SelenePlus** para G6, G8, G10 y G12 con profundidad \(p=3\) y
512 shots por grafo.

El trabajo ya ejecutado es:

`a0ee7567-39f4-4816-a0eb-227624bdce85`

La recuperación de ese trabajo no consume nuevos recursos. El envío de un
nuevo trabajo está desactivado por defecto y requiere cambiar explícitamente
`ENVIAR_NUEVO_JOB` a `True`.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import ast
import json

import numpy as np
import pandas as pd
from IPython.display import display


def localizar_raiz():
    candidatos = [Path.cwd(), Path.cwd().parent]
    for candidato in candidatos:
        if (
            (candidato / "datos" / "grafos").exists()
            and (candidato / "resultados" / "qaoa_local").exists()
        ):
            return candidato.resolve()
    raise FileNotFoundError(
        "Ejecute el notebook desde notebooks/ o desde la raíz "
        "de reto1_qaoa_limpio."
    )


ROOT = localizar_raiz()
GRAFOS = ROOT / "datos" / "grafos"
QAOA_LOCAL = ROOT / "resultados" / "qaoa_local"
REMOTO = ROOT / "resultados" / "remoto"
EVIDENCIA = REMOTO / "evidencia_seleneplus"
EVIDENCIA.mkdir(parents=True, exist_ok=True)

TAMANOS_SELENE = [6, 8, 10, 12]
P_SELENE = 3
SHOTS_SELENE = 512
SEED_SELENE = 2026
JOB_ID_SELENE = "a0ee7567-39f4-4816-a0eb-227624bdce85"
PROJECT_ID_SELENE = "8274481c-a9c6-41b7-b34d-5f4ab74192b3"
ENVIAR_NUEVO_JOB = False

print("Raíz:", ROOT)
print("Job existente:", JOB_ID_SELENE)
print("Enviar trabajo nuevo:", ENVIAR_NUEVO_JOB)


Raíz: /home/jovyan/reto1_qaoa_noir
Job existente: a0ee7567-39f4-4816-a0eb-227624bdce85
Enviar trabajo nuevo: False


## 1. Carga de grafos y parámetros optimizados

Se conserva el orden de nodos de los CSV definitivos y se selecciona el mejor
reinicio ideal de \(p=3\) previamente calculado en
`03_qaoa_ideal_guppy.ipynb`.


In [2]:
def a_lista_float(valor):
    if isinstance(valor, str):
        texto = valor.strip()
        if texto.startswith("array(") and texto.endswith(")"):
            texto = texto[6:-1]
        try:
            valor = ast.literal_eval(texto)
        except (ValueError, SyntaxError):
            valor = np.fromstring(
                texto.replace("[", "").replace("]", "").replace(",", " "),
                sep=" ",
            )
    salida = np.asarray(valor, dtype=float).reshape(-1)
    assert np.isfinite(salida).all()
    return [float(x) for x in salida]


def cargar_grafo(n):
    nodos = pd.read_csv(GRAFOS / f"grafo_{n}_nodos.csv")
    aristas = pd.read_csv(GRAFOS / f"grafo_{n}_aristas.csv")
    ids = nodos["node_id"].astype(str).str.strip().tolist()
    assert len(ids) == n and len(set(ids)) == n
    indice = {node_id: i for i, node_id in enumerate(ids)}
    columna_source = "source" if "source" in aristas else "source_id"
    columna_target = "target" if "target" in aristas else "target_id"
    lista = [
        (
            int(indice[str(fila[columna_source]).strip()]),
            int(indice[str(fila[columna_target]).strip()]),
            float(fila["weight"]),
        )
        for _, fila in aristas.iterrows()
    ]
    assert len(lista) == n - 1
    return {"nodos": nodos, "aristas_df": aristas, "aristas": lista}


parametros = pd.read_csv(QAOA_LOCAL / "mejores_parametros_qaoa.csv")
configuraciones = {}

for n in TAMANOS_SELENE:
    fila = parametros.loc[
        parametros["n"].eq(n) & parametros["p"].eq(P_SELENE)
    ]
    assert len(fila) == 1, f"No existe una única configuración G{n}, p=3."
    fila = fila.iloc[0]
    grafo = cargar_grafo(n)
    configuraciones[n] = {
        **grafo,
        "gammas": a_lista_float(fila["gammas_fisicos"]),
        "betas": a_lista_float(fila["betas"]),
    }
    assert len(configuraciones[n]["gammas"]) == P_SELENE
    assert len(configuraciones[n]["betas"]) == P_SELENE

display(pd.DataFrame([
    {
        "grafo": f"G{n}",
        "n": n,
        "aristas": len(configuraciones[n]["aristas"]),
        "p": P_SELENE,
        "gammas": configuraciones[n]["gammas"],
        "betas": configuraciones[n]["betas"],
    }
    for n in TAMANOS_SELENE
]))


,grafo,n,aristas,p,gammas,betas
0,G6,6,5,3,"[0.07952650846421255, 0.1370744662633656, 0.15...","[0.5982047480202383, 0.45898147025636066, 0.29..."
1,G8,8,7,3,"[0.07757595472153715, 0.13582782857573483, 0.1...","[2.1586015042784155, 1.986191237234369, 1.8305..."
2,G10,10,9,3,"[0.03610147913864077, 0.07014521620481483, 0.0...","[2.1327026822716824, 1.8829327178435242, 0.160..."
3,G12,12,11,3,"[0.036031062597456365, 0.06734416728822452, 0....","[0.526293468215285, 0.26592697199368337, 1.713..."


## 2. Construcción y compilación Guppy

Los ángulos optimizados están en radianes. Para las operaciones angulares de
Guppy se expresan como múltiplos de `pi`. Cada programa tiene un entrypoint sin
argumentos y devuelve las mediciones en el registro `c`.


In [3]:
import qnexus as qnx
from guppylang import guppy
from guppylang.std.angles import pi
from guppylang.std.builtins import array, comptime, result
from guppylang.std.quantum import h, measure_array, qubit, rx

try:
    from guppylang.std.qsystem import zz_phase
except ImportError:
    from guppylang.std.qsystem.helios import zz_phase


def construir_qaoa_guppy(n, aristas, gammas, betas):
    aristas_ct = [
        (int(i), int(j), float(peso))
        for i, j, peso in aristas
    ]
    capas_ct = [
        (float(gamma) / np.pi, float(beta) / np.pi)
        for gamma, beta in zip(gammas, betas)
    ]

    @guppy
    def main_qaoa() -> None:
        qs = array(qubit() for _ in range(comptime(n)))
        for i in range(comptime(n)):
            h(qs[i])

        for gamma_pi, beta_pi in comptime(capas_ct):
            for i, j, peso in comptime(aristas_ct):
                zz_phase(qs[i], qs[j], -gamma_pi * peso * pi)
            for i in range(comptime(n)):
                rx(qs[i], 2.0 * beta_pi * pi)

        result("c", measure_array(qs))

    main_qaoa.check()
    return main_qaoa


programas_guppy = {
    n: construir_qaoa_guppy(
        n,
        configuraciones[n]["aristas"],
        configuraciones[n]["gammas"],
        configuraciones[n]["betas"],
    )
    for n in TAMANOS_SELENE
}

hugr_selene = {
    n: programas_guppy[n].compile()
    for n in TAMANOS_SELENE
}

print("Programas Guppy verificados y compilados a HUGR:", TAMANOS_SELENE)


Programas Guppy verificados y compilados a HUGR: [6, 8, 10, 12]


## 3. Configuración SelenePlus utilizada

- `StatevectorSimulator(seed=2026)`
- `HeliosRuntime(seed=2026)`
- `QSystemErrorModel(seed=2026, name="alpha")`
- 512 shots por cada uno de los cuatro programas


In [4]:
CONFIG_SELENE = qnx.models.SelenePlusConfig(
    simulator=qnx.models.StatevectorSimulator(seed=SEED_SELENE),
    runtime=qnx.models.HeliosRuntime(seed=SEED_SELENE),
    error_model=qnx.models.QSystemErrorModel(
        seed=SEED_SELENE,
        name="alpha",
    ),
)

print("Configuración:", type(CONFIG_SELENE).__name__)
print("Simulador:", type(CONFIG_SELENE.simulator).__name__)
print("Runtime:", type(CONFIG_SELENE.runtime).__name__)
print("Modelo de error:", type(CONFIG_SELENE.error_model).__name__)
print("Grafos:", TAMANOS_SELENE)
print("Profundidad:", P_SELENE)
print("Shots por grafo:", SHOTS_SELENE)


Configuración: SelenePlusConfig
Simulador: StatevectorSimulator
Runtime: HeliosRuntime
Modelo de error: QSystemErrorModel
Grafos: [6, 8, 10, 12]
Profundidad: 3
Shots por grafo: 512


## 4. Evidencia del trabajo original ejecutado

La salida preservada a continuación pertenece a la ejecución original. Muestra
el identificador del trabajo, proyecto, fecha de creación y envío, configuración
completa, cuatro elementos y que `wait_for()` retornó antes de obtener cuatro
referencias de resultado.


In [5]:
# EVIDENCIA PRESERVADA. Esta celda NO vuelve a enviar el trabajo.
print(
    "Job original:",
    JOB_ID_SELENE,
)
print(
    "La salida histórica preservada debajo corresponde a "
    "start_execute_job(...), cuatro elementos, wait_for(...) "
    "y cuatro resultados disponibles."
)


Job original: a0ee7567-39f4-4816-a0eb-227624bdce85
La salida histórica preservada debajo corresponde a start_execute_job(...), cuatro elementos, wait_for(...) y cuatro resultados disponibles.


## 5. Recuperación del job existente y resultados

Esta celda inicia sesión y recupera el trabajo por UUID mediante
`qnx.jobs.get(id=...)`. No crea ni ejecuta un trabajo nuevo.


In [6]:
qnx.login()
job_selene = qnx.jobs.get(id=JOB_ID_SELENE)

print("Job recuperado:", job_selene.id)
print("Estado actualizado:", job_selene.last_status)
print("Nombre:", job_selene.annotations.name)
print("Creado:", job_selene.annotations.created)
print("Detalle de estado:", job_selene.last_status_detail)

assert str(job_selene.id) == JOB_ID_SELENE
assert str(job_selene.last_status).upper().endswith("COMPLETED")

resultados_ref = qnx.jobs.results(job_selene)
assert len(resultados_ref) == 4
print("Resultados recuperados:", len(resultados_ref))


Already logged in. Tokens are valid.
Job recuperado: a0ee7567-39f4-4816-a0eb-227624bdce85
Estado actualizado: JobStatusEnum.COMPLETED
Nombre: QAOA-G6-G8-G10-G12-p3-SelenePlus-20260723-195058
Creado: 2026-07-23 19:50:59.971956+00:00
Detalle de estado: JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 7, 23, 19, 51, 19, 72207, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 7, 23, 19, 51, 5, 591269, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 7, 23, 19, 50, 59, 981025, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 7, 23, 19, 51, 7, 49994, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)
Resultados recuperados: 4


## 6. Evaluación y exportación de conteos completos

Los resultados se asocian con G6, G8, G10 y G12 usando los IDs de los HUGR
originales. Se verifican exactamente 512 shots y la longitud de cada bitstring.


In [7]:
HUGR_ID_A_N = {
    "2ac5be14-b839-46bd-b37a-43d2162ffd90": 6,
    "4d6fafef-3c77-446b-ac6a-d2f615c120e1": 8,
    "955711c4-afd3-490d-a366-4e51def2343f": 10,
    "34962464-f590-463c-adb1-208f9e7ab007": 12,
}


def costo_maxcut(bitstring, aristas):
    bits = [int(x) for x in str(bitstring)]
    return float(sum(
        peso for i, j, peso in aristas
        if bits[i] != bits[j]
    ))


resumen_ideal = pd.read_csv(
    QAOA_LOCAL / "resumen_mediciones_512_shots.csv"
)
filas_conteos = []
filas_resumen = []

for resultado_ref in resultados_ref:
    hugr_id = str(resultado_ref.get_input().id)
    assert hugr_id in HUGR_ID_A_N, f"HUGR desconocido: {hugr_id}"
    n = HUGR_ID_A_N[hugr_id]
    resultado = resultado_ref.download_result()
    registros = resultado.register_counts(
        strict_names=True,
        strict_lengths=True,
    )
    assert "c" in registros
    conteos = registros["c"]
    shots = int(sum(conteos.values()))
    assert shots == SHOTS_SELENE
    assert all(len(str(bits)) == n for bits in conteos)

    optimo = float(sum(
        peso for _, _, peso in configuraciones[n]["aristas"]
    ))
    costo_esperado = 0.0
    prob_optimo = 0.0
    mejor_costo = 0.0

    for bitstring, frecuencia in conteos.items():
        costo = costo_maxcut(
            bitstring,
            configuraciones[n]["aristas"],
        )
        probabilidad = int(frecuencia) / shots
        costo_esperado += probabilidad * costo
        mejor_costo = max(mejor_costo, costo)
        if np.isclose(costo, optimo, atol=1e-6):
            prob_optimo += probabilidad
        filas_conteos.append({
            "job_id": JOB_ID_SELENE,
            "hugr_id": hugr_id,
            "grafo": f"G{n}",
            "n": n,
            "p": P_SELENE,
            "shots_configurados": SHOTS_SELENE,
            "registro": "c",
            "bitstring": str(bitstring),
            "frecuencia": int(frecuencia),
            "probabilidad": probabilidad,
            "costo": costo,
        })

    ideal = resumen_ideal.loc[
        resumen_ideal["n"].eq(n)
        & resumen_ideal["p"].eq(P_SELENE)
    ].iloc[0]
    razon_selene = costo_esperado / optimo
    razon_ideal = float(ideal["razon_ideal"])
    filas_resumen.append({
        "job_id": JOB_ID_SELENE,
        "estado": "COMPLETED",
        "backend": "SelenePlus",
        "grafo": f"G{n}",
        "n": n,
        "p": P_SELENE,
        "shots": shots,
        "costo_esperado": costo_esperado,
        "razon_ideal": razon_ideal,
        "razon_seleneplus": razon_selene,
        "error_absoluto": abs(razon_selene - razon_ideal),
        "mejor_costo": mejor_costo,
        "optimo_exacto": optimo,
        "probabilidad_optimo": prob_optimo,
    })

df_conteos = pd.DataFrame(filas_conteos).sort_values(
    ["n", "frecuencia", "bitstring"],
    ascending=[True, False, True],
)
df_resumen_selene = pd.DataFrame(filas_resumen).sort_values("n")
df_verificacion = (
    df_conteos.groupby(["job_id", "grafo", "n", "p"], as_index=False)
    .agg(
        shots_recibidos=("frecuencia", "sum"),
        estados_observados=("bitstring", "nunique"),
    )
)

assert set(df_resumen_selene["n"]) == set(TAMANOS_SELENE)
assert df_verificacion["shots_recibidos"].eq(SHOTS_SELENE).all()

df_conteos.to_csv(
    EVIDENCIA / "seleneplus_conteos_completos.csv",
    index=False,
)
df_verificacion.to_csv(
    EVIDENCIA / "seleneplus_verificacion_shots.csv",
    index=False,
)
df_resumen_selene.to_csv(
    REMOTO / "seleneplus_p3_resumen.csv",
    index=False,
)

display(df_verificacion)
display(df_resumen_selene.round(6))
print("Conteos y resumen exportados en:", EVIDENCIA)


,job_id,grafo,n,p,shots_recibidos,estados_observados
0,a0ee7567-39f4-4816-a0eb-227624bdce85,G10,10,3,512,185
1,a0ee7567-39f4-4816-a0eb-227624bdce85,G12,12,3,512,342
2,a0ee7567-39f4-4816-a0eb-227624bdce85,G6,6,3,512,29
3,a0ee7567-39f4-4816-a0eb-227624bdce85,G8,8,3,512,77


,job_id,estado,backend,grafo,n,p,shots,costo_esperado,razon_ideal,razon_seleneplus,error_absoluto,mejor_costo,optimo_exacto,probabilidad_optimo
0,a0ee7567-39f4-4816-a0eb-227624bdce85,COMPLETED,SelenePlus,G6,6,3,512,25.764369,0.917346,0.672668,0.244678,36.334644,38.301790,0.0
1,a0ee7567-39f4-4816-a0eb-227624bdce85,COMPLETED,SelenePlus,G8,8,3,512,21.872843,0.894451,0.518092,0.376358,41.538491,42.218037,0.0
2,a0ee7567-39f4-4816-a0eb-227624bdce85,COMPLETED,SelenePlus,G10,10,3,512,44.811944,0.871222,0.514636,0.356587,84.428356,87.075048,0.0
3,a0ee7567-39f4-4816-a0eb-227624bdce85,COMPLETED,SelenePlus,G12,12,3,512,46.706827,0.853893,0.500611,0.353282,89.067897,93.299621,0.0


Conteos y resumen exportados en: /home/jovyan/reto1_qaoa_noir/resultados/remoto/evidencia_seleneplus


## 7. Exportación JSON y metadatos auditables

La fecha terminal se toma del estado actualizado devuelto por Nexus. Si no
estuviera disponible, se conserva como `null`; nunca se inventa.


In [8]:
conteos_json = {
    "job_id": JOB_ID_SELENE,
    "programas": {},
}

for n in TAMANOS_SELENE:
    parte = df_conteos.loc[df_conteos["n"].eq(n)]
    conteos_json["programas"][f"G{n}"] = {
        "n": n,
        "p": P_SELENE,
        "shots": int(parte["frecuencia"].sum()),
        "registro": "c",
        "conteos": {
            str(fila.bitstring): int(fila.frecuencia)
            for fila in parte.itertuples(index=False)
        },
    }

with open(
    EVIDENCIA / "seleneplus_resultados_crudos.json",
    "w",
    encoding="utf-8",
) as archivo:
    json.dump(conteos_json, archivo, indent=2, ensure_ascii=False)

detalle = job_selene.last_status_detail
metadatos = {
    "job_id": JOB_ID_SELENE,
    "job_name": job_selene.annotations.name,
    "project_id": PROJECT_ID_SELENE,
    "project_name": "QAOA Max-Cut con Selene",
    "status": str(job_selene.last_status).split(".")[-1],
    "created_utc": (
        job_selene.annotations.created.isoformat()
        if job_selene.annotations.created else None
    ),
    "submitted_time_utc": (
        detalle.submitted_time.isoformat()
        if detalle.submitted_time else None
    ),
    "running_time_utc": (
        detalle.running_time.isoformat()
        if detalle.running_time else None
    ),
    "completed_time_utc": (
        detalle.completed_time.isoformat()
        if detalle.completed_time else None
    ),
    "n_resultados": len(resultados_ref),
    "tamanos": TAMANOS_SELENE,
    "p": P_SELENE,
    "shots_por_resultado": SHOTS_SELENE,
    "shots_totales": SHOTS_SELENE * len(TAMANOS_SELENE),
    "backend_config": {
        "type": "SelenePlusConfig",
        "simulator": {"type": "StatevectorSimulator", "seed": SEED_SELENE},
        "runtime": {"type": "HeliosRuntime", "seed": SEED_SELENE},
        "error_model": {
            "type": "QSystemErrorModel",
            "seed": SEED_SELENE,
            "name": "alpha",
        },
    },
    "hugr_ids": {
        f"G{n}": hugr_id
        for hugr_id, n in HUGR_ID_A_N.items()
    },
    "exported_utc": datetime.now(timezone.utc).isoformat(),
}

with open(
    EVIDENCIA / "seleneplus_metadatos_job.json",
    "w",
    encoding="utf-8",
) as archivo:
    json.dump(metadatos, archivo, indent=2, ensure_ascii=False)

display(pd.Series(metadatos, name="valor").to_frame())
print("Evidencia primaria exportada correctamente.")


,valor
job_id,a0ee7567-39f4-4816-a0eb-227624bdce85
job_name,QAOA-G6-G8-G10-G12-p3-SelenePlus-20260723-195058
project_id,8274481c-a9c6-41b7-b34d-5f4ab74192b3
project_name,QAOA Max-Cut con Selene
status,COMPLETED
created_utc,2026-07-23T19:50:59.971956+00:00
submitted_time_utc,2026-07-23T19:50:59.981025+00:00
running_time_utc,2026-07-23T19:51:07.049994+00:00
completed_time_utc,2026-07-23T19:51:19.072207+00:00
n_resultados,4


Evidencia primaria exportada correctamente.


## 8. Envío opcional de un nuevo trabajo — desactivado

Esta celda solo se ejecuta si el usuario cambia conscientemente
`ENVIAR_NUEVO_JOB = True`. El flujo normal de auditoría usa el job existente y
no entra en este bloque.


In [9]:
if not ENVIAR_NUEVO_JOB:
    print(
        "Envío omitido: se reutiliza el job existente",
        JOB_ID_SELENE,
    )
else:
    qnx.login()
    proyecto = qnx.projects.get_or_create(
        name="QAOA Max-Cut con Selene",
    )
    qnx.context.set_active_project(proyecto)
    sufijo = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    hugr_refs = {
        n: qnx.hugr.upload(
            hugr_package=hugr_selene[n],
            name=f"QAOA-G{n}-p{P_SELENE}-{sufijo}",
            project=proyecto,
        )
        for n in TAMANOS_SELENE
    }
    nuevo_job = qnx.start_execute_job(
        programs=[hugr_refs[n] for n in TAMANOS_SELENE],
        n_shots=[SHOTS_SELENE] * len(TAMANOS_SELENE),
        n_qubits=TAMANOS_SELENE,
        backend_config=CONFIG_SELENE,
        project=proyecto,
        name=(
            f"QAOA-G6-G8-G10-G12-p{P_SELENE}-"
            f"SelenePlus-{sufijo}"
        ),
    )
    print(
        "Nuevo trabajo enviado explícitamente. "
        "Registre su UUID antes de continuar:",
        nuevo_job,
    )


Envío omitido: se reutiliza el job existente a0ee7567-39f4-4816-a0eb-227624bdce85


## Conclusión

El notebook conserva el flujo reproducible de construcción Guppy, la
configuración de SelenePlus, la salida histórica del trabajo original y una
ruta segura para recuperar los cuatro resultados existentes. Al ejecutar las
secciones 5–7 en Nexus se generan conteos completos, verificación de shots,
resultado crudo JSON y metadatos originales actualizados sin enviar un trabajo
nuevo.
